In [2]:
import json
import os
import re


# ==========================================
# 1. STORAGE LAYER (JSON Persistence)
# ==========================================

def load_contacts(filepath):
    # Agar file abhi exist nahi karti (pehli baar run ho raha hai) to khali dict dein
    if not os.path.exists(filepath):
        return {}

    try:
        with open(filepath, "r", encoding="utf-8") as file:
            content = file.read().strip()
            if content == "":
                return {}
            data = json.loads(content)
    except (json.JSONDecodeError, OSError) as err:
        # Corrupt file ko overwrite ya silently delete hone se bachane ke liye error raise karein
        raise RuntimeError(f"Corrupt ya unreadable JSON file '{filepath}': {err}")

    if type(data) != dict:
        raise RuntimeError("JSON file ka root format dictionary hona chahiye")

    return data


def save_contacts(filepath, contacts):
    # Safe write: pehle temp file mein likhein, phir original file ko replace karein
    temp_path = filepath + ".tmp"
    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(contacts, file, indent=2)
    os.replace(temp_path, filepath)


# ==========================================
# 2. BUSINESS LOGIC & VALIDATION LAYER
# ==========================================

def validate_contact_fields(contact_id, name, phone, email):
    cid = str(contact_id).strip()
    if not cid:
        raise ValueError("Contact ID khali nahi ho sakti")

    clean_name = str(name).strip()
    if not clean_name:
        raise ValueError("Name khali nahi ho sakta")

    clean_phone = str(phone).strip()
    if not clean_phone or not re.match(r"^\+?[0-9\-\s]{7,15}$", clean_phone):
        raise ValueError("Invalid phone number (7-15 digits allowed)")

    clean_email = str(email).strip()
    if not clean_email or not re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", clean_email):
        raise ValueError("Invalid email address")

    return cid, clean_name, clean_phone, clean_email


def add_contact(filepath, contact_id, name, phone, email):
    cid, clean_name, clean_phone, clean_email = validate_contact_fields(
        contact_id, name, phone, email
    )
    contacts = load_contacts(filepath)

    if cid in contacts:
        raise ValueError(f"Contact ID '{cid}' pehle se maujood hai")

    contacts[cid] = {
        "id": cid,
        "name": clean_name,
        "phone": clean_phone,
        "email": clean_email
    }
    save_contacts(filepath, contacts)
    return contacts[cid]


def search_contacts(filepath, query):
    if type(query) != str or not query.strip():
        raise ValueError("Search query khali nahi ho sakti")

    contacts = load_contacts(filepath)
    keyword = query.strip().lower()
    matches = []

    for cid, info in contacts.items():
        if keyword in info["name"].lower():
            matches.append(dict(info))

    return matches


def update_contact(filepath, contact_id, name=None, phone=None, email=None):
    cid = str(contact_id).strip()
    contacts = load_contacts(filepath)

    if cid not in contacts:
        raise KeyError(f"Contact ID '{cid}' nahi mili")

    existing = contacts[cid]
    new_name = existing["name"] if name is None or str(name).strip() == "" else name
    new_phone = existing["phone"] if phone is None or str(phone).strip() == "" else phone
    new_email = existing["email"] if email is None or str(email).strip() == "" else email

    _, clean_name, clean_phone, clean_email = validate_contact_fields(
        cid, new_name, new_phone, new_email
    )

    contacts[cid] = {
        "id": cid,
        "name": clean_name,
        "phone": clean_phone,
        "email": clean_email
    }
    save_contacts(filepath, contacts)
    return contacts[cid]


def delete_contact(filepath, contact_id):
    cid = str(contact_id).strip()
    contacts = load_contacts(filepath)

    if cid not in contacts:
        raise KeyError(f"Contact ID '{cid}' nahi mili")

    deleted = contacts.pop(cid)
    save_contacts(filepath, contacts)
    return deleted


def list_contacts(filepath):
    contacts = load_contacts(filepath)
    return [dict(info) for info in contacts.values()]


# ==========================================
# 3. MENU-DRIVEN CLI LAYER
# ==========================================

def run_cli(filepath="contacts.json"):
    while True:
        print("\n--- Contact Manager ---")
        print("1. Add Contact")
        print("2. Search Contact by Name")
        print("3. Update Contact")
        print("4. Delete Contact")
        print("5. List All Contacts")
        print("6. Exit")

        choice = input("Choose an option (1-6): ").strip()

        try:
            if choice == "1":
                cid = input("Enter unique ID: ")
                name = input("Enter name: ")
                phone = input("Enter phone: ")
                email = input("Enter email: ")
                record = add_contact(filepath, cid, name, phone, email)
                print("Contact added:", record)

            elif choice == "2":
                query = input("Enter name to search: ")
                results = search_contacts(filepath, query)
                if len(results) == 0:
                    print("Koi matching contact nahi mila.")
                else:
                    for item in results:
                        print(item)

            elif choice == "3":
                cid = input("Enter ID to update: ")
                print("(Khali chhor dein agar field change nahi karni)")
                name = input("New name: ")
                phone = input("New phone: ")
                email = input("New email: ")
                updated = update_contact(filepath, cid, name, phone, email)
                print("Contact updated:", updated)

            elif choice == "4":
                cid = input("Enter ID to delete: ")
                removed = delete_contact(filepath, cid)
                print("Contact deleted:", removed)

            elif choice == "5":
                all_items = list_contacts(filepath)
                if len(all_items) == 0:
                    print("Contact list khali hai.")
                else:
                    for item in all_items:
                        print(item)

            elif choice == "6":
                print("Exiting Contact Manager.")
                break

            else:
                print("Invalid choice. 1 se 6 ke darmiyan number select karein.")

        except (ValueError, KeyError, RuntimeError) as err:
            print("Error:", err)


if __name__ == "__main__":
    # Sample verification (persistent restart test)
    demo_file = "sample_contacts.json"
    if not os.path.exists(demo_file):
        add_contact(demo_file, "1", "Ayesha", "03001234567", "ayesha@example.com")

    print("Sample Search ('aye'):", search_contacts(demo_file, "aye"))
    run_cli(demo_file)

Sample Search ('aye'): [{'id': '1', 'name': 'Ayesha', 'phone': '03001234567', 'email': 'ayesha@example.com'}]

--- Contact Manager ---
1. Add Contact
2. Search Contact by Name
3. Update Contact
4. Delete Contact
5. List All Contacts
6. Exit


Koi matching contact nahi mila.

--- Contact Manager ---
1. Add Contact
2. Search Contact by Name
3. Update Contact
4. Delete Contact
5. List All Contacts
6. Exit
Invalid choice. 1 se 6 ke darmiyan number select karein.

--- Contact Manager ---
1. Add Contact
2. Search Contact by Name
3. Update Contact
4. Delete Contact
5. List All Contacts
6. Exit
Exiting Contact Manager.
